In [1]:
import numpy as np
from libow8 import sensor_net
import matplotlib.pyplot as plt
import owutils as ut
from designs_diag import designs, align_receiver_to_transmitter
from panel_ow import Panel

In [2]:
import numpy as np
import cma
import sys
import multiprocessing as mp

# ==========================================
# 1. GLOBAL SETUP & PARAMETERS
# ==========================================


KEY = 'A1'
params_d = designs[KEY]
params_amb = designs['A1'].copy()
params_amb['r_master'] = params_d['r_lights']
params_amb['PT_master'] = params_d['PT_lights']

# ==========================================
# 2. WORKER FUNCTION FOR MULTIPROCESSING
# ==========================================
def optimize_single_sensor(args):
    """
    Worker function to optimize a single sensor. 
    Runs entirely in its own process memory space.
    """
    sensor_idx, r_sens, nS_sens, p_d, p_amb = args
    
    
    state = {
        'h_ww': None,
        'h_amb': None
    }

    
    def sensor_ar_vectorized(x_pop):
        n_particles = x_pop.shape[0]
        theta = x_pop[:, 0]
        phi   = x_pop[:, 1]
        area  = x_pop[:, 2]

        r_sensor = np.tile(r_sens, (n_particles, 1))
        nS_sensor = nS_sens 
        nR = ut.spher_to_cart_ar(1, theta, phi).T

        
        local_p_amb = p_amb.copy()
        local_p_d = p_d.copy()

        local_p_amb['r_sensor'] = r_sensor
        local_p_amb['nR_sensor'] = nR
        local_p_amb['A_sensor'] = area   
        local_p_amb['nS_sensor'] = nS_sensor    

        local_p_d['r_sensor'] = r_sensor
        local_p_d['nR_sensor'] = nR
        local_p_d['A_sensor'] = area    
        local_p_d['nS_sensor'] = nS_sensor   
        
        # Ambient Engine
        l_amb = sensor_net(**local_p_amb) 
        l_amb.calch(h_ww=state['h_amb'])
        l_amb.light_sim()
        if state['h_amb'] is None: 
            state['h_amb'] = l_amb.h_ww
        
        # Signal Engine
        l = sensor_net(**local_p_d) 
        l.calch(h_ww=state['h_ww'])
        l.light_sim()
        if state['h_ww'] is None:
            state['h_ww'] = l.h_ww        
        
        l.calc_noise() 
        l.calc_rq() 

        p_all = np.sum(np.sum(l_amb.Pin_sm_diff, axis=0), axis=1) + np.sum(l_amb.Pin_sm, axis=0) + l_amb.Pin_sa + 0.5*l.Pin_sm_tot.flatten()
        G_ac = l.Pin_sm_tot.flatten() / area
        G_all = p_all / area
        
        snr_dB = np.zeros(n_particles)
        e_day = np.zeros(n_particles)
        bitrates = np.zeros(n_particles) 
        
        freq = np.linspace(100, 20000, 50)

        for i in range(n_particles):
            p = Panel(area[i]*1e4, rs=1, rsh=1000, n=1.6, voc=0.64, isc=35e-3, G=G_all[i], Gac=G_ac[i])
            p.run(False)
            
            ind = int(p.ind)
            p.calc_capacitance()
            p.set_circuit(Rc=10, Lo=100e-6, Co=1000e-6)
            p.find_bw()
            
            p.tf(freq)
            p.all_thermal_noise(freq)
            p.shot_noise(freq)
            p.vp2p(freq)
            
            sig = p.vac[ind]
            noise = 4 * (p.th_noise[ind] + p.sh_noise[ind])
            
            if noise <= 1e-20 or sig <= 1e-20:
                snr_dB[i] = -100.0
            else:
                snr_dB[i] = 10 * np.log10( (sig**2) / noise )
            
            e_day[i] = p.Pmax * 3600 * 8 * 0.8
            bitrates[i] = np.floor(p.BW[ind]) * 0.4
        
        l.calc_tbattery(br=bitrates)

        V = 3.3
        cycle_p = np.array([c.calc_cycle_consumption() for c in l.cycles]) * V
        day_s = 3600 * 24
        cycles_day = day_s / l.Tcycle
        e_day_c = cycles_day * cycle_p

        penalty = np.zeros(n_particles)
        
        mask_snr = snr_dB < 8.5
        penalty[mask_snr] += (8.5 - snr_dB[mask_snr]) * 10000
        
        mask_eng = e_day < e_day_c
        penalty[mask_eng] += (e_day_c[mask_eng] - e_day[mask_eng]) * 10000
        
        return area + penalty

    # --- Nested Fitness Wrapper ---
    def fit_function_batch(x_norm_batch):
        theta = x_norm_batch[:,0] * (np.pi/2)
        phi   = x_norm_batch[:,1] * (2*np.pi)
        area  = 0.0005 + x_norm_batch[:,2] * (0.03 - 0.0005)
        x_denorm = np.column_stack((theta, phi, area))
        return sensor_ar_vectorized(x_denorm)

    # --- CMA-ES Execution ---
    print(f"[Process Started] Sensor {sensor_idx} at {r_sens}...")
    sys.stdout.flush()

    es = cma.CMAEvolutionStrategy(3 * [0.5], 0.25, {
        'bounds': [0, 1], 
        'popsize': 80, 
        'verbose': -9, 
        'maxiter': 80
    })
    
    while not es.stop():
        solutions = es.ask()
        X_batch = np.array(solutions)
        fitness_values = fit_function_batch(X_batch)
        es.tell(solutions, fitness_values)
        
    best_x_norm = es.result.xbest
    theta = best_x_norm[0] * (np.pi/2)
    phi   = best_x_norm[1] * (2*np.pi)
    area  = 0.0005 + best_x_norm[2] * (0.03 - 0.0005)
    final_cost = es.result.fbest
    
    print(f"[Process Finished] Sensor {sensor_idx} -> Area={area*1e4:.2f} cm2 | Cost={final_cost:.4f}")
    sys.stdout.flush()

    return {
        'pos': r_sens,
        'theta': np.degrees(theta),
        'phi': np.degrees(phi),
        'area_cm2': area * 1e4,
        'cost': final_cost
    }

# ==========================================
# 3. MAIN EXECUTION BLOCK (MULTIPROCESSING)
# ==========================================

if __name__ == '__main__':
    r_sen_list = designs[KEY]['r_sensor']
    nS_sen_list = np.round(align_receiver_to_transmitter(r_sen_list, np.array([5,5,3])), 2)
    N_sensors = r_sen_list.shape[0]

    print(f"Starting Vectorized CMA-ES for {N_sensors} sensors on 4 Processors...")
    
    
    tasks = []
    for i in range(N_sensors):
        
        tasks.append((i+1, r_sen_list[i], nS_sen_list[i], params_d.copy(), params_amb.copy()))

    # Initialize a pool with 4 processors
    with mp.Pool(processes=4) as pool:
        cma_simple_C = pool.map(optimize_single_sensor, tasks)

    # ==========================================
    # 4. SAVE RESULTS TO .NPY
    # ==========================================
    
    results_array = np.array(cma_simple_C)
    
    
    filename = 'cma_simple_A1_eight.npy'
    np.save(filename, results_array)
    
    print("-" * 60)
    print("Optimization Complete.")
    print(f"Results successfully saved to '{filename}'.")
    

Starting Vectorized CMA-ES for 21 sensors on 4 Processors...
[Process Started] Sensor 3 at [1.36 1.36 0.  ]...[Process Started] Sensor 5 at [2.27 2.27 0.  ]...[Process Started] Sensor 1 at [0.45 0.45 0.  ]...[Process Started] Sensor 7 at [3.18 3.18 0.  ]...





/home/alexandros/Επιφάνεια εργασίας/eh-ow-main (x)/eh-ow-main_8/libow8.py:42: RuntimeWarning: invalid value encountered in divide
  self.un_n1 = self.dr1 / self.abs_dr1
/home/alexandros/Επιφάνεια εργασίας/eh-ow-main (x)/eh-ow-main_8/libow8.py:42: RuntimeWarning: invalid value encountered in divide
  self.un_n1 = self.dr1 / self.abs_dr1
/home/alexandros/Επιφάνεια εργασίας/eh-ow-main (x)/eh-ow-main_8/libow8.py:42: RuntimeWarning: invalid value encountered in divide
  self.un_n1 = self.dr1 / self.abs_dr1
/home/alexandros/Επιφάνεια εργασίας/eh-ow-main (x)/eh-ow-main_8/libow8.py:42: RuntimeWarning: invalid value encountered in divide
  self.un_n1 = self.dr1 / self.abs_dr1
/home/alexandros/Επιφάνεια εργασίας/eh-ow-main (x)/eh-ow-main_8/libow8.py:43: RuntimeWarning: invalid value encountered in divide
  self.un_n2 = self.dr2 / self.abs_dr2
/home/alexandros/Επιφάνεια εργασίας/eh-ow-main (x)/eh-ow-main_8/libow8.py:43: RuntimeWarning: invalid value encountered in divide
  self.un_n2 = self.dr2 /

[Process Finished] Sensor 5 -> Area=57.01 cm2 | Cost=0.0057
[Process Started] Sensor 6 at [2.73 2.73 0.  ]...
[Process Finished] Sensor 3 -> Area=78.77 cm2 | Cost=0.0079
[Process Started] Sensor 4 at [1.82 1.82 0.  ]...
[Process Finished] Sensor 7 -> Area=44.66 cm2 | Cost=0.0045
[Process Started] Sensor 8 at [3.64 3.64 0.  ]...
[Process Finished] Sensor 1 -> Area=122.97 cm2 | Cost=0.0123
[Process Started] Sensor 2 at [0.91 0.91 0.  ]...
[Process Finished] Sensor 4 -> Area=67.79 cm2 | Cost=0.0068
[Process Started] Sensor 9 at [4.09 4.09 0.  ]...
[Process Finished] Sensor 6 -> Area=50.33 cm2 | Cost=0.0050
[Process Started] Sensor 11 at [5. 5. 0.]...
[Process Finished] Sensor 8 -> Area=40.99 cm2 | Cost=0.0041
[Process Started] Sensor 13 at [5.91 5.91 0.  ]...
[Process Finished] Sensor 2 -> Area=95.91 cm2 | Cost=0.0096
[Process Started] Sensor 15 at [6.82 6.82 0.  ]...
[Process Finished] Sensor 9 -> Area=38.66 cm2 | Cost=0.0039
[Process Started] Sensor 10 at [4.55 4.55 0.  ]...
[Process Fi